# 🚀 Indian Equity Trading System - Google Colab Edition

**A comprehensive short-term trading system for Indian equity markets (NSE/BSE) with 5-day trading horizon**

## Features:
- ✅ Advanced Technical Indicators (Yang-Zhang volatility, Supertrend, Ichimoku, KST, etc.)
- ✅ Candlestick Pattern Recognition (7 patterns with success rates)
- ✅ Machine Learning Models (Random Forest, XGBoost with purged CV)
- ✅ Backtesting Framework (with Indian transaction costs)
- ✅ Portfolio Management (Multiple position sizing strategies)
- ✅ Signal Generation & Stock Ranking

---

**⚠️ Disclaimer:** This is for educational purposes only. Always paper trade first!

## 📦 Step 1: Installation & Setup

This will install all required dependencies. Takes ~2-3 minutes.

In [ ]:
# Install dependencies
!pip install -q yfinance pandas numpy scikit-learn xgboost imbalanced-learn numba plotly matplotlib seaborn pyarrow tqdm

print("✅ All dependencies installed successfully!")

## 📥 Step 2: Clone Repository & Setup

Clone the trading system code from GitHub.

In [ ]:
import os

# Clone repository if not already cloned
if not os.path.exists('letssee'):
    !git clone https://github.com/harshitsingh85420/letssee.git
    print("✅ Repository cloned!")
else:
    print("✅ Repository already exists!")

# Change to project directory
%cd /content/letssee/indian_trading_system

# Add to Python path
import sys
sys.path.insert(0, '/content/letssee/indian_trading_system')

print("✅ Setup complete! Ready to trade 📈")

## 🔧 Step 3: Import Modules

Import all the trading system components.

In [ ]:
# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Data modules
from data.loader import DataLoader
from data.cleaner import DataCleaner

# Indicator modules
from indicators.technical import TechnicalIndicators
from indicators.volatility import VolatilityEstimators
from indicators.patterns import CandlestickPatterns

# ML modules
from models.features import FeatureEngineer
from models.ml_models import MLModels

# Portfolio modules
from portfolio.manager import PortfolioManager
from portfolio.signals import SignalGenerator, StockRanker

# Backtesting
from backtesting.engine import BacktestEngine

# Utils
from utils.constants import TOP_10_NIFTY, NIFTY_50_SYMBOLS
from utils.indian_market import IndianMarketUtils, PerformanceMetrics

# For visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ All modules imported successfully!")
print(f"\n📊 Available stocks: {TOP_10_NIFTY[:5]}... (Top 10 NIFTY)")

---

# 🎯 Quick Examples

## Example 1: Load & Visualize Stock Data

Let's start by loading data for a stock and visualizing it.

In [ ]:
# Initialize data loader
loader = DataLoader()

# Load data for Reliance
symbol = 'RELIANCE.NS'
print(f"📥 Loading data for {symbol}...")

df = loader.load_stock_data(symbol)

if df is not None:
    print(f"✅ Loaded {len(df)} days of data")
    print(f"📅 Date range: {df['date'].min()} to {df['date'].max()}")
    
    # Display basic info
    print("\n📊 Recent data:")
    display(df[['date', 'open', 'high', 'low', 'close', 'volume']].tail(10))
    
    # Plot price chart
    fig = go.Figure()
    
    fig.add_trace(go.Candlestick(
        x=df['date'],
        open=df['open'],
        high=df['high'],
        low=df['low'],
        close=df['close'],
        name='OHLC'
    ))
    
    fig.update_layout(
        title=f'{symbol} - Price Chart',
        yaxis_title='Price (₹)',
        xaxis_title='Date',
        height=500
    )
    
    fig.show()
    
    # Volume chart
    fig2 = go.Figure()
    fig2.add_trace(go.Bar(x=df['date'], y=df['volume'], name='Volume'))
    fig2.update_layout(
        title=f'{symbol} - Volume Chart',
        yaxis_title='Volume',
        xaxis_title='Date',
        height=300
    )
    fig2.show()
else:
    print("❌ Failed to load data")

## Example 2: Calculate Technical Indicators

Calculate and visualize all technical indicators.

In [ ]:
# Calculate indicators
print("📊 Calculating technical indicators...")

technical = TechnicalIndicators()
volatility = VolatilityEstimators()
patterns = CandlestickPatterns()

# Add all indicators
df_indicators = technical.calculate_all(df)
df_indicators = volatility.calculate_all(df_indicators)
df_indicators = patterns.detect_all_patterns(df_indicators)
df_indicators = patterns.calculate_pattern_strength(df_indicators)

print("✅ Indicators calculated!")

# Display recent indicators
indicator_cols = ['date', 'close', 'supertrend_direction', 'adx', 'kst', 'cmf', 'yang_zhang_vol', 'pattern_strength']
print("\n📈 Recent indicators:")
display(df_indicators[indicator_cols].tail(10))

# Plot Supertrend
fig = make_subplots(rows=3, cols=1, 
                    shared_xaxes=True,
                    vertical_spacing=0.05,
                    subplot_titles=('Price with Supertrend', 'ADX (Trend Strength)', 'Volume'),
                    row_heights=[0.5, 0.25, 0.25])

# Price and Supertrend
fig.add_trace(go.Scatter(x=df_indicators['date'], y=df_indicators['close'], 
                         name='Close', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=df_indicators['date'], y=df_indicators['supertrend'], 
                         name='Supertrend', line=dict(color='red', dash='dash')), row=1, col=1)

# ADX
fig.add_trace(go.Scatter(x=df_indicators['date'], y=df_indicators['adx'], 
                         name='ADX', line=dict(color='purple')), row=2, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1)

# Volume
fig.add_trace(go.Bar(x=df_indicators['date'], y=df_indicators['volume'], 
                     name='Volume'), row=3, col=1)

fig.update_layout(height=800, title_text=f"{symbol} - Technical Analysis", showlegend=True)
fig.show()

# Pattern summary
pattern_summary = patterns.get_pattern_summary(df_indicators)
if not pattern_summary.empty:
    print("\n🕯️ Candlestick Patterns Detected:")
    display(pattern_summary)

## Example 3: Train Machine Learning Models

Train Random Forest and XGBoost models with cross-validation.

In [ ]:
print("🤖 Training ML Models...\n")

# Feature engineering
engineer = FeatureEngineer()
df_features = engineer.create_all_features(df_indicators)
X, y, feature_names = engineer.prepare_ml_data(df_features)

print(f"\n📊 Dataset Info:")
print(f"  Samples: {len(X)}")
print(f"  Features: {len(feature_names)}")
print(f"  Positive samples: {y.sum()} ({y.mean():.1%})")
print(f"  Negative samples: {len(y) - y.sum()} ({1-y.mean():.1%})")

if len(X) > 100:
    # Initialize ML models
    ml_models = MLModels()
    
    # Cross-validation (using fewer folds for speed in Colab)
    print("\n🔄 Running Cross-Validation...")
    print("\n--- Random Forest ---")
    rf_cv = ml_models.cross_validate(X, y, model_type='rf')
    
    print("\n--- XGBoost ---")
    xgb_cv = ml_models.cross_validate(X, y, model_type='xgb')
    
    # Train final models
    print("\n🎯 Training final models...")
    ml_models.train_final_models(X, y, feature_names)
    
    # Feature importance
    print("\n📊 Top 15 Most Important Features:")
    importance = ml_models.get_feature_importance(top_n=15)
    display(importance)
    
    # Plot feature importance
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=importance['avg_importance'],
        y=importance['feature'],
        orientation='h',
        marker_color='steelblue'
    ))
    fig.update_layout(
        title='Top 15 Feature Importance',
        xaxis_title='Importance',
        yaxis_title='Feature',
        height=500
    )
    fig.show()
    
    # Make predictions
    predictions = ml_models.predict(X, model_type='ensemble')
    df_features['ml_prediction'] = predictions
    
    print(f"\n✅ ML training complete!")
    print(f"Average prediction probability: {predictions.mean():.3f}")
else:
    print("⚠️ Not enough data for ML training (need >100 samples)")

## Example 4: Generate Trading Signals

Combine all indicators and ML to generate trading signals.

In [ ]:
print("🎯 Generating Trading Signals...\n")

signal_gen = SignalGenerator()

# Get ML predictions if available
ml_pred = df_features['ml_prediction'] if 'ml_prediction' in df_features.columns else None

# Generate signals
df_signals = signal_gen.generate_all_signals(df_indicators, ml_pred)

print("✅ Signals generated!\n")

# Show recent signals
print("📊 Recent Trading Signals (Last 15 days):")
signal_summary = signal_gen.get_signal_summary(df_signals, recent_days=15)
display(signal_summary)

# Plot composite signal
fig = make_subplots(rows=2, cols=1,
                    shared_xaxes=True,
                    vertical_spacing=0.05,
                    subplot_titles=('Price', 'Composite Signal'),
                    row_heights=[0.6, 0.4])

# Price
fig.add_trace(go.Scatter(x=df_signals['date'], y=df_signals['close'],
                         name='Close', line=dict(color='blue')), row=1, col=1)

# Mark buy/sell signals
buy_signals = df_signals[df_signals['trading_signal'] == 1]
sell_signals = df_signals[df_signals['trading_signal'] == -1]

fig.add_trace(go.Scatter(x=buy_signals['date'], y=buy_signals['close'],
                         mode='markers', name='Buy Signal',
                         marker=dict(color='green', size=10, symbol='triangle-up')),
              row=1, col=1)

fig.add_trace(go.Scatter(x=sell_signals['date'], y=sell_signals['close'],
                         mode='markers', name='Sell Signal',
                         marker=dict(color='red', size=10, symbol='triangle-down')),
              row=1, col=1)

# Composite signal
fig.add_trace(go.Scatter(x=df_signals['date'], y=df_signals['composite_signal'],
                         name='Composite Signal', line=dict(color='purple'),
                         fill='tozeroy'), row=2, col=1)

fig.add_hline(y=0.3, line_dash="dash", line_color="green", row=2, col=1)
fig.add_hline(y=-0.3, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=0, line_dash="solid", line_color="gray", row=2, col=1)

fig.update_layout(height=700, title_text=f"{symbol} - Trading Signals", showlegend=True)
fig.show()

# Signal statistics
total_buy = len(buy_signals)
total_sell = len(sell_signals)
print(f"\n📈 Signal Statistics:")
print(f"  Total Buy Signals: {total_buy}")
print(f"  Total Sell Signals: {total_sell}")
print(f"  Current Signal: {df_signals['trading_signal'].iloc[-1]}")
print(f"  Current Composite Score: {df_signals['composite_signal'].iloc[-1]:.3f}")

## Example 5: Backtest Strategy

Run a complete backtest with Indian market transaction costs.

In [ ]:
print("📊 Running Backtest...\n")

# Initialize backtest engine
engine = BacktestEngine(initial_capital=1000000)  # 10 Lakhs

# Run backtest
results = engine.run_backtest(
    df_signals,
    df_signals['trading_signal'],
    position_size=0.3  # 30% position size
)

# Print detailed results
engine.print_results(results)

# Plot equity curve
fig = go.Figure()

equity_curve = results['equity_curve']
fig.add_trace(go.Scatter(x=df_signals['date'], y=equity_curve,
                         name='Portfolio Value', line=dict(color='green', width=2),
                         fill='tonexty'))

fig.add_hline(y=results['initial_capital'], line_dash="dash", 
              line_color="blue", annotation_text="Initial Capital")

fig.update_layout(
    title=f'{symbol} - Backtest Equity Curve',
    yaxis_title='Portfolio Value (₹)',
    xaxis_title='Date',
    height=500
)
fig.show()

# Drawdown chart
cummax = equity_curve.cummax()
drawdown = (equity_curve - cummax) / cummax * 100

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df_signals['date'], y=drawdown,
                          name='Drawdown', line=dict(color='red'),
                          fill='tozeroy'))
fig2.update_layout(
    title=f'{symbol} - Drawdown',
    yaxis_title='Drawdown (%)',
    xaxis_title='Date',
    height=400
)
fig2.show()

# Trade analysis
if 'trades' in results and not results['trades'].empty:
    trades_df = results['trades']
    print("\n💰 Recent Trades:")
    display(trades_df.tail(10))
    
    # P&L distribution
    sell_trades = trades_df[trades_df['type'] == 'SELL']
    if not sell_trades.empty and 'pnl' in sell_trades.columns:
        fig3 = go.Figure()
        fig3.add_trace(go.Histogram(x=sell_trades['pnl'], nbinsx=30,
                                    marker_color='steelblue'))
        fig3.update_layout(
            title='Trade P&L Distribution',
            xaxis_title='P&L (₹)',
            yaxis_title='Frequency',
            height=400
        )
        fig3.show()

## Example 6: Multi-Stock Analysis & Portfolio

Analyze multiple stocks and create a portfolio.

In [ ]:
print("📊 Multi-Stock Analysis\n")
print(f"Analyzing: {TOP_10_NIFTY}\n")

# Load data for multiple stocks
print("📥 Loading data for top 10 NIFTY stocks...")
data_dict = loader.load_multiple_stocks(TOP_10_NIFTY)
print(f"✅ Loaded {len(data_dict)} stocks\n")

# Calculate indicators and signals for all stocks
print("🔄 Processing stocks...")
signals_dict = {}

for symbol, stock_df in data_dict.items():
    try:
        # Calculate indicators
        stock_df = technical.calculate_all(stock_df)
        stock_df = volatility.calculate_all(stock_df)
        stock_df = patterns.detect_all_patterns(stock_df)
        stock_df = patterns.calculate_pattern_strength(stock_df)
        
        # Generate signals
        stock_df = signal_gen.generate_all_signals(stock_df)
        
        signals_dict[symbol] = stock_df
        print(f"  ✅ {symbol}")
    except Exception as e:
        print(f"  ❌ {symbol}: {str(e)}")

# Rank stocks by signal strength
print("\n🏆 Stock Rankings (by Signal Strength):\n")
ranker = StockRanker()
rankings = ranker.rank_by_signal_strength(signals_dict, top_n=10)
display(rankings)

# Visualize top 5 stocks
top_5 = rankings.head(5)['symbol'].tolist()

fig = go.Figure()
for symbol in top_5:
    stock_df = signals_dict[symbol]
    fig.add_trace(go.Scatter(x=stock_df['date'], y=stock_df['close'],
                             name=symbol.replace('.NS', ''),
                             mode='lines'))

fig.update_layout(
    title='Top 5 Stocks - Price Comparison (Last 6 Months)',
    yaxis_title='Normalized Price',
    xaxis_title='Date',
    height=500
)
fig.show()

# Portfolio simulation
print("\n💼 Running Portfolio Backtest...")
data_for_backtest = {k: v for k, v in signals_dict.items()}
signals_for_backtest = {k: v['trading_signal'] for k, v in signals_dict.items()}

portfolio_results = engine.run_multi_stock_backtest(
    data_for_backtest,
    signals_for_backtest,
    position_size_per_stock=0.1  # 10% per stock
)

engine.print_results(portfolio_results)

# Plot portfolio equity
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=portfolio_results['equity_curve'].index,
                          y=portfolio_results['equity_curve'].values,
                          name='Portfolio Value',
                          line=dict(color='green', width=2),
                          fill='tonexty'))

fig2.update_layout(
    title='Multi-Stock Portfolio - Equity Curve',
    yaxis_title='Portfolio Value (₹)',
    xaxis_title='Date',
    height=500
)
fig2.show()

## Example 7: Transaction Cost Analysis

Analyze Indian market transaction costs.

In [ ]:
print("💰 Indian Market Transaction Cost Analysis\n")

market_utils = IndianMarketUtils()

# Calculate costs for different trade sizes
trade_sizes = [50000, 100000, 250000, 500000, 1000000]
cost_data = []

for size in trade_sizes:
    costs = market_utils.calculate_transaction_costs(size)
    cost_data.append({
        'Trade Size (₹)': f"₹{size:,}",
        'Brokerage': f"₹{costs['brokerage']:.2f}",
        'STT': f"₹{costs['stt']:.2f}",
        'GST': f"₹{costs['gst']:.2f}",
        'Total Round-Trip': f"₹{costs['total_round_trip']:.2f}",
        'Cost %': f"{costs['total_round_trip_pct']:.4f}%"
    })

cost_df = pd.DataFrame(cost_data)
display(cost_df)

# Plot cost percentage vs trade size
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[f"₹{s/100000:.1f}L" for s in trade_sizes],
    y=[market_utils.calculate_transaction_costs(s)['total_round_trip_pct'] for s in trade_sizes],
    mode='lines+markers',
    name='Transaction Cost %',
    line=dict(color='red', width=2),
    marker=dict(size=10)
))

fig.update_layout(
    title='Transaction Costs vs Trade Size',
    xaxis_title='Trade Size',
    yaxis_title='Cost (%)',
    height=400
)
fig.show()

print("\n💡 Key Insights:")
print("  • Brokerage capped at ₹20 per trade")
print("  • STT is 0.1% on sell side only")
print("  • Total round-trip cost: ~0.3-0.5%")
print("  • Larger trades have lower percentage costs")

---

# 🚀 Complete Trading System Workflow

Run the complete trading system end-to-end.

In [ ]:
from main import TradingSystem

print("="*80)
print("🚀 RUNNING COMPLETE TRADING SYSTEM WORKFLOW")
print("="*80)
print()

# Create trading system
system = TradingSystem(
    symbols=TOP_10_NIFTY[:5],  # Use top 5 for faster processing in Colab
    initial_capital=1000000
)

# Run complete workflow
results = system.run_complete_workflow(
    train_ml=True,   # Train ML models
    backtest=True    # Run backtest
)

print("\n" + "="*80)
print("✅ WORKFLOW COMPLETE!")
print("="*80)

# Display results
print("\n🏆 TOP TRADING OPPORTUNITIES:")
display(results['rankings'])

# Show backtest summary
if results['backtest_results']:
    br = results['backtest_results']
    print(f"\n📊 PORTFOLIO PERFORMANCE:")
    print(f"  Initial Capital: ₹{br['initial_capital']:,.2f}")
    print(f"  Final Equity: ₹{br['final_equity']:,.2f}")
    print(f"  Total Return: {br['total_return']:.2%}")
    if 'sharpe_ratio' in br:
        print(f"  Sharpe Ratio: {br['sharpe_ratio']:.2f}")
    if 'max_drawdown' in br:
        print(f"  Max Drawdown: {br['max_drawdown']:.2%}")

print("\n🎉 System ready for trading!")

---

# 🔧 Custom Analysis Section

Use this section to run your own custom analysis.

In [ ]:
# YOUR CUSTOM CODE HERE

# Example: Analyze a specific stock
my_symbol = 'TCS.NS'

# Load and analyze
my_df = loader.load_stock_data(my_symbol)
if my_df is not None:
    print(f"Analyzing {my_symbol}...")
    
    # Add your analysis here
    # ...
    
    print("Analysis complete!")

## 💾 Save Results to Google Drive (Optional)

Mount Google Drive to save your results permanently.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create directory for results
import os
save_dir = '/content/drive/MyDrive/indian_trading_system_results'
os.makedirs(save_dir, exist_ok=True)

# Save rankings
if 'rankings' in results and results['rankings'] is not None:
    results['rankings'].to_csv(f"{save_dir}/stock_rankings.csv", index=False)
    print(f"✅ Saved rankings to {save_dir}/stock_rankings.csv")

# Save backtest results
if 'backtest_results' in results and results['backtest_results']:
    br = results['backtest_results']
    if 'trades' in br and br['trades'] is not None:
        br['trades'].to_csv(f"{save_dir}/backtest_trades.csv", index=False)
        print(f"✅ Saved trades to {save_dir}/backtest_trades.csv")

print("\n📁 All results saved to Google Drive!")

---

# 🎓 Next Steps

## 📚 What You've Learned:
1. ✅ Loading and cleaning Indian stock data
2. ✅ Calculating advanced technical indicators
3. ✅ Detecting candlestick patterns
4. ✅ Training ML models for prediction
5. ✅ Generating trading signals
6. ✅ Backtesting strategies with real costs
7. ✅ Multi-stock portfolio management

## 🚀 Recommended Actions:
1. **Paper Trade First** - Never use real money until thoroughly tested
2. **Customize Parameters** - Adjust in `utils/constants.py`
3. **Add More Stocks** - Expand beyond TOP_10_NIFTY
4. **Monitor Performance** - Track live vs backtest results
5. **Iterate and Improve** - Refine based on results

## ⚠️ Important Reminders:
- This is for **educational purposes only**
- **Past performance ≠ future results**
- Always use **stop losses**
- Never risk more than you can afford to lose
- Consider consulting a financial advisor

## 📖 Documentation:
- Full documentation in `README.md`
- Setup guide in `SETUP.md`
- Code examples in `example_usage.py`

---

**Happy Trading! 📈💰**

*Built with ❤️ using Claude Code*